# **RETO 1:**

In [10]:
import pandas as pd
import json
from datetime import datetime


# Nombres de las hojas del Excel
HOJA_EMPLEADOS = 'Empleados Activos'
HOJA_INCAPACIDADES = 'Incapacidades'
HOJA_EXAMENES = 'Examenes Medicos'

# ID del cliente (fijo para BPO Soluciones)
CLIENTE_ID = 'bpo-soluciones-001'

# PASO 1: CARGAR DATOS DEL EXCEL

archivo_excel = '/content/datos_cliente_bpo_soluciones_1.xlsx'
def cargar_datos(archivo_excel):

    try:
        empleados = pd.read_excel(archivo_excel, sheet_name=HOJA_EMPLEADOS)
        incapacidades = pd.read_excel(archivo_excel, sheet_name=HOJA_INCAPACIDADES)
        examenes = pd.read_excel(archivo_excel, sheet_name=HOJA_EXAMENES)

        print(f"{len(empleados)} empleados cargados exitosamente")
        print(f"{len(incapacidades)} incapacidades cargadas exitosamente")
        print(f"{len(examenes)} exámenes cargados exitosamente")

        return empleados, incapacidades, examenes

    except Exception as e:
        print(f"❌ Error al cargar el Excel: {e}")
        raise


# PASO 2: LIMPIEZA Y NORMALIZACIÓN DE DATOS

def limpiar_texto(texto):
    """
    Normaliza un texto: mayúsculas, sin espacios extra, sin tildes.

    ¿Por qué?
    - Los nombres pueden venir con formato inconsistente
    - Cosmos DB es case-sensitive, necesitamos consistencia
    """
    if pd.isna(texto) or texto == '':
        return None

    # Convertir a string y mayúsculas
    texto = str(texto).upper().strip()

    # Quitar tildes con la librería unicodedata
    import unicodedata
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

    return texto

def limpiar_empleados(df):
    """
    Limpia y valida el DataFrame de empleados.

    Validaciones:
    - Cédula no puede ser nula (es el ID)
    - Textos normalizados a mayúsculas
    """
    # Convertir cédula a string limpio (sin decimales)
    df['CEDULA'] = df['CEDULA'].astype(str).str.split('.').str[0]

    # Eliminar empleados sin cédula
    antes = len(df)
    df = df.dropna(subset=['CEDULA'])
    if len(df) < antes:
        print(f"Eliminados {antes - len(df)} empleados sin cédula")

    # Normalizar textos
    columnas_texto = ['PRIMER NOMBRE', 'SEGUNDO NOMBRE', 'PRIMER APELLIDO',
                      'SEGUNDO APELLIDO', 'CARGO', 'TIPO CONTRATO', 'AREA', 'ESTADO', 'CIUDAD']

    for col in columnas_texto:
        if col in df.columns:
            df[col] = df[col].apply(limpiar_texto)

    print(f"{len(df)} empleados validados")
    return df

def limpiar_incapacidades(df):
    # La columna se llama 'CEDULA EMPLEADO' en esta hoja
    df['CEDULA'] = df['CEDULA EMPLEADO'].astype(str).str.split('.').str[0]
    df = df.dropna(subset=['CEDULA'])

    # Convertir DIAS a número entero
    df['DIAS'] = pd.to_numeric(df['DIAS'], errors='coerce').fillna(0).astype(int)

    print(f"{len(df)} incapacidades validadas")
    return df

def limpiar_examenes(df):

    df['CEDULA'] = df['CEDULA'].astype(str).str.split('.').str[0]
    df = df.dropna(subset=['CEDULA'])

    print(f"{len(df)} exámenes validados")
    return df

# PASO 3: CONSTRUIR DOCUMENTOS COSMOS

def construir_nombres(row):
    """
    Construye el objeto 'nombres' según estructura Cosmos.

    ¿Por qué una función separada?
    - Cada objeto anidado merece su propia función
    - Fácil de testear individualmente
    - Claro qué campos van en cada parte
    """
    return {
        "primerNombre": row.get('PRIMER NOMBRE'),
        "segundoNombre": row.get('SEGUNDO NOMBRE'),
        "primerApellido": row.get('PRIMER APELLIDO'),
        "segundoApellido": row.get('SEGUNDO APELLIDO')
    }

def construir_info_laboral(row):
    """Construye el objeto 'infoLaboral'."""
    return {
        "cargo": row.get('CARGO'),
        "area": row.get('AREA'),
        "fechaIngreso": row.get('FECHA INGRESO'),
        "tipoContrato": row.get('TIPO CONTRATO'),
        "estado": row.get('ESTADO')
    }

def construir_contacto(row):
    """Construye el objeto 'contacto'."""
    return {
        "correo": row.get('CORREO CORPORATIVO'),
        "telefono": str(row.get('TELEFONO', '')) if pd.notna(row.get('TELEFONO')) else None,
        "ciudad": row.get('CIUDAD')
    }

def construir_incapacidad(row):
    """
    Construye un objeto de incapacidad con su diagnóstico anidado.

    Estructura:
    {
      "fechaInicio": "...",
      "diasIncapacidad": 14,
      "diagnostico": {
        "codigoCIE10": "...",
        "descripcion": "..."
      }
    }
    """
    return {
        "fechaInicio": row.get('FECHA INICIO INCAPACIDAD'),
        "fechaFin": row.get('FECHA FIN INCAPACIDAD'),
        "diasIncapacidad": int(row.get('DIAS', 0)),
        "tipoIncapacidad": limpiar_texto(row.get('TIPO INCAPACIDAD')),
        "diagnostico": {
            "codigoCIE10": row.get('DIAGNOSTICO CIE10'),
            "descripcion": limpiar_texto(row.get('DESCRIPCION DIAGNOSTICO'))
        },
        "entidad": limpiar_texto(row.get('ENTIDAD'))
    }

def construir_examen(row):
    """
    Construye un objeto de examen médico.

    Nota sobre RESTRICCIONES:
    - Puede venir como texto separado por comas
    - Lo convertimos a lista
    - Si está vacío, lista vacía []
    """
    restricciones_texto = row.get('RESTRICCIONES', '')

    if pd.isna(restricciones_texto) or restricciones_texto == '':
        restricciones = []
    else:
        # Separar por comas y limpiar
        restricciones = [r.strip() for r in str(restricciones_texto).split(',')]

    return {
        "tipoExamen": row.get('TIPO EXAMEN'),
        "fechaExamen": row.get('FECHA EXAMEN'),
        "resultado": row.get('RESULTADO'),
        "restricciones": restricciones,
        "proximaFecha": row.get('PROXIMA FECHA'),
        "medico": row.get('MEDICO')
    }

def construir_documento_empleado(empleado_row, incapacidades_empleado, examenes_empleado):
    """
    Construye el documento completo de un empleado para Cosmos DB.

    ¿Por qué esta estructura?
    - Cosmos DB necesita un campo 'id' único (usamos cédula)
    - 'clienteId' es la partition key (todos los empleados del mismo cliente)
    - Incapacidades y exámenes van anidados dentro del documento
    """
    cedula = empleado_row['CEDULA']

    # Construir nombre completo
    nombres = [
        empleado_row.get('PRIMER NOMBRE'),
        empleado_row.get('SEGUNDO NOMBRE'),
        empleado_row.get('PRIMER APELLIDO'),
        empleado_row.get('SEGUNDO APELLIDO')
    ]
    # Convertir a string y filtrar valores vacíos/None/nan
    nombre_completo = ' '.join([
        str(n) for n in nombres
        if n and str(n) not in ['None', 'nan', '']
    ]).strip()

    documento = {
        "id": cedula,
        "cedula": cedula,
        "nombres": construir_nombres(empleado_row),
        "nombreCompleto": nombre_completo,
        "infoLaboral": construir_info_laboral(empleado_row),
        "contacto": construir_contacto(empleado_row),
        "clienteId": CLIENTE_ID,
        "tipo": "empleado",
        "fechaCreacion": datetime.now().isoformat() + 'Z',
        "fechaActualizacion": datetime.now().isoformat() + 'Z',
        "incapacidades": incapacidades_empleado,
        "examenesMedicos": examenes_empleado
    }

    return documento

# =============================================================================
# PASO 4: PIPELINE PRINCIPAL
# =============================================================================

def ejecutar_pipeline(archivo_excel, archivo_salida='empleados_cosmos.json'):
    """
    Pipeline completo: Excel -> Cosmos JSON

    Flujo:
    1. Cargar datos
    2. Limpiar y validar
    3. Agrupar incapacidades y exámenes por empleado
    4. Construir documentos
    5. Exportar JSON
    """
    print("="*80)
    print("INICIANDO PIPELINE DE INGESTION")
    print("="*80)

    # PASO 1: Cargar
    empleados, incapacidades, examenes = cargar_datos(archivo_excel)

    # PASO 2: Limpiar
    empleados = limpiar_empleados(empleados)
    incapacidades = limpiar_incapacidades(incapacidades)
    examenes = limpiar_examenes(examenes)

    # PASO 3: Agrupar datos por cédula
    print("\nAgrupando incapacidades y exámenes por empleado...")

    # Crear diccionarios: cedula -> lista de incapacidades/examenes
    incap_por_cedula = {}
    for cedula, grupo in incapacidades.groupby('CEDULA'):
        incap_por_cedula[cedula] = [
            construir_incapacidad(row)
            for _, row in grupo.iterrows()
        ]

    exam_por_cedula = {}
    for cedula, grupo in examenes.groupby('CEDULA'):
        exam_por_cedula[cedula] = [
            construir_examen(row)
            for _, row in grupo.iterrows()
        ]

    # PASO 4: Construir documentos
    print("\nConstruyendo documentos Cosmos...")
    documentos = []

    for _, empleado in empleados.iterrows():
        cedula = empleado['CEDULA']

        # Obtener incapacidades y exámenes (si no hay, lista vacía)
        incapacidades_emp = incap_por_cedula.get(cedula, [])
        examenes_emp = exam_por_cedula.get(cedula, [])

        doc = construir_documento_empleado(empleado, incapacidades_emp, examenes_emp)
        documentos.append(doc)

    # PASO 5: Exportar
    print(f"\nExportando {len(documentos)} documentos a {archivo_salida}...")

    with open(archivo_salida, 'w', encoding='utf-8') as f:
        json.dump(documentos, f, indent=2, ensure_ascii=False, default=str)

    print("\n" + "="*80)
    print("✅ PIPELINE COMPLETADO EXITOSAMENTE")
    print("="*80)
    print(f"\n📊 Resumen:")
    print(f"   - Empleados procesados: {len(documentos)}")
    print(f"   - Incapacidades totales: {len(incapacidades)}")
    print(f"   - Exámenes totales: {len(examenes)}")
    print(f"   - Archivo generado: {archivo_salida}")

    return documentos




## **Implementación de la solución generada**

In [4]:
# =============================================================================
# EJECUCIÓN
# =============================================================================

if __name__ == "__main__":
    # Ruta al archivo Excel
    ARCHIVO_EXCEL = '/content/datos_cliente_bpo_soluciones_1.xlsx'

    # Ejecutar pipeline
    documentos = ejecutar_pipeline(ARCHIVO_EXCEL)

    # Mostrar ejemplo del primer documento
    print("\n📄 Ejemplo del primer documento generado:")
    print(json.dumps(documentos[0], indent=2, ensure_ascii=False))

INICIANDO PIPELINE DE INGESTION
300 empleados cargados exitosamente
526 incapacidades cargadas exitosamente
473 exámenes cargados exitosamente
300 empleados validados
526 incapacidades validadas
473 exámenes validados

Agrupando incapacidades y exámenes por empleado...

Construyendo documentos Cosmos...

Exportando 300 documentos a empleados_cosmos.json...

✅ PIPELINE COMPLETADO EXITOSAMENTE

📊 Resumen:
   - Empleados procesados: 300
   - Incapacidades totales: 526
   - Exámenes totales: 473
   - Archivo generado: empleados_cosmos.json

📄 Ejemplo del primer documento generado:
{
  "id": "95822412",
  "cedula": "95822412",
  "nombres": {
    "primerNombre": "DIANA",
    "segundoNombre": "FELIPE",
    "primerApellido": "AGUILAR",
    "segundoApellido": "VEGA"
  },
  "nombreCompleto": "DIANA FELIPE AGUILAR VEGA",
  "infoLaboral": {
    "cargo": "LIDER DE EQUIPO",
    "area": "ADMINISTRACION",
    "fechaIngreso": "2019-07-26",
    "tipoContrato": "PRESTACION DE SERVICIOS",
    "estado": "A

## **Prueba de salida del archivo JSON**

In [5]:
# Leer el JSON generado
with open('empleados_cosmos.json', 'r') as f:
    documentos = json.load(f)

# Buscar un empleado que tenga incapacidades y exámenes
empleado_ejemplo = None
for doc in documentos:
    if len(doc.get('incapacidades', [])) > 0 and len(doc.get('examenesMedicos', [])) > 0:
        empleado_ejemplo = doc
        break

print("📄 Ejemplo de documento completo con incapacidades y exámenes:")
print(json.dumps(empleado_ejemplo, indent=2, ensure_ascii=False)[:2000])
print("\n...")
print(f"\n✅ Total de documentos generados: {len(documentos)}")
print(f"✅ Empleados con incapacidades: {sum(1 for d in documentos if len(d.get('incapacidades', [])) > 0)}")
print(f"✅ Empleados con exámenes: {sum(1 for d in documentos if len(d.get('examenesMedicos', [])) > 0)}")

📄 Ejemplo de documento completo con incapacidades y exámenes:
{
  "id": "95822412",
  "cedula": "95822412",
  "nombres": {
    "primerNombre": "DIANA",
    "segundoNombre": "FELIPE",
    "primerApellido": "AGUILAR",
    "segundoApellido": "VEGA"
  },
  "nombreCompleto": "DIANA FELIPE AGUILAR VEGA",
  "infoLaboral": {
    "cargo": "LIDER DE EQUIPO",
    "area": "ADMINISTRACION",
    "fechaIngreso": "2019-07-26",
    "tipoContrato": "PRESTACION DE SERVICIOS",
    "estado": "ACTIVO"
  },
  "contacto": {
    "correo": "diana.aguilar@bposoluciones.com",
    "telefono": "3283197857",
    "ciudad": "BOGOTA"
  },
  "clienteId": "bpo-soluciones-001",
  "tipo": "empleado",
  "fechaCreacion": "2026-02-13T02:58:03.196194Z",
  "fechaActualizacion": "2026-02-13T02:58:03.196211Z",
  "incapacidades": [
    {
      "fechaInicio": "2022-04-24",
      "fechaFin": "2022-05-02",
      "diasIncapacidad": 8,
      "tipoIncapacidad": "ENFERMEDAD LABORAL",
      "diagnostico": {
        "codigoCIE10": "H669",


# **RETO 2**

In [6]:

import pandas as pd
import json

def analizar_discrepancia():
    """
    Analiza la discrepancia entre el conteo de empleados activos
    reportado por el cliente vs lo que muestra la plataforma.

    Metodología:
    1. Contar empleados ACTIVOS en Excel original
    2. Contar empleados ACTIVOS en JSON generado
    3. Identificar diferencias
    4. Proponer corrección
    """

    print("="*80)
    print("ANÁLISIS DE DISCREPANCIA - EMPLEADOS ACTIVOS")
    print("="*80)

    # PASO 1: Analizar Excel del cliente
    print("\nPASO 1: Analizando Excel del cliente...")
    df = pd.read_excel('/content/datos_cliente_bpo_soluciones_1.xlsx', sheet_name='Empleados Activos')

    print(f"Total de empleados en Excel: {len(df)}")

    # Ver valores EXACTOS del campo ESTADO (sin normalizar)
    print(f"\nValores en campo ESTADO:")
    print(df['ESTADO'].value_counts())

    # Normalizar para análisis
    df['ESTADO_NORMALIZADO'] = df['ESTADO'].str.upper().str.strip()

    activos_excel = df[df['ESTADO_NORMALIZADO'] == 'ACTIVO']
    inactivos_excel = df[df['ESTADO_NORMALIZADO'] == 'INACTIVO']

    print(f"\nEmpleados ACTIVOS: {len(activos_excel)}")
    print(f"Empleados INACTIVOS: {len(inactivos_excel)}")

    # PASO 2: Analizar JSON generado (lo que ve la plataforma)
    print("\nPASO 2: Analizando JSON generado (plataforma Narah)...")

    with open('empleados_cosmos.json', 'r') as f:
        documentos = json.load(f)

    activos_cosmos = [
        d for d in documentos
        if d.get('infoLaboral', {}).get('estado') == 'ACTIVO'
    ]

    inactivos_cosmos = [
        d for d in documentos
        if d.get('infoLaboral', {}).get('estado') == 'INACTIVO'
    ]

    print(f"Empleados ACTIVOS en plataforma: {len(activos_cosmos)}")
    print(f"Empleados INACTIVOS en plataforma: {len(inactivos_cosmos)}")

    # PASO 3: Identificar discrepancia
    print("\nPASO 3: Identificando discrepancia...")

    discrepancia = len(activos_excel) - len(activos_cosmos)

    if discrepancia == 0:
        print("No hay discrepancia. Conteo coincide.")
    else:
        print(f"       DISCREPANCIA DETECTADA:")
        print(f"      Excel del cliente: {len(activos_excel)} activos")
        print(f"      Plataforma Narah:  {len(activos_cosmos)} activos")
        print(f"      Diferencia:        {abs(discrepancia)} empleado(s)")

    # PASO 4: Buscar la causa raíz
    print("\nPASO 4: Buscando causa raíz...")

    # Posibles causas:
    # 1. Empleado marcado como ACTIVO pero con variación en el texto
    # 2. Empleado con estado NULL/vacío
    # 3. Empleado duplicado

    # Verificar variaciones de "ACTIVO"
    print("\nVerificando variaciones en el texto 'ACTIVO':")
    for valor in df['ESTADO'].unique():
        count = len(df[df['ESTADO'] == valor])
        valor_norm = str(valor).upper().strip()
        if 'ACTIVO' in valor_norm and valor_norm != 'INACTIVO':
            print(f"      '{valor}' (normalizado: '{valor_norm}') -> {count} empleados")

    # Buscar empleados con estado problemático
    estados_validos = ['ACTIVO', 'INACTIVO']
    problematicos = df[~df['ESTADO_NORMALIZADO'].isin(estados_validos)]

    if len(problematicos) > 0:
        print(f"\nEncontrados {len(problematicos)} empleados con estado inválido:")
        print(problematicos[['CEDULA', 'PRIMER NOMBRE', 'PRIMER APELLIDO', 'ESTADO']])

    # PASO 6: Proponer corrección
    print("\n💡 PASO 6: Propuesta de corrección:")
    print("""
    CAUSA PROBABLE:
    - El pipeline normalizó correctamente el campo ESTADO a mayúsculas
    - Si el Excel tenía variaciones como "Activo ", " ACTIVO", etc.,
      el pipeline las corrigió a "ACTIVO"
    - La discrepancia reportada por el cliente puede deberse a que ellos
      cuentan manualmente sin normalizar

    CORRECCIÓN:
    1. Validar con el cliente que sus 78 empleados incluyan SOLO
       registros con estado exactamente = "ACTIVO"
    2. Revisar si hay empleados que ellos consideran activos pero que
       en el Excel están marcados como "INACTIVO"
    3. Si se confirma un error de marcación:
       - Actualizar el Excel
       - Re-ejecutar el pipeline
       - Verificar el conteo

    PREVENCIÓN:
    - Agregar validación al pipeline que reporte empleados con estados
      no estándar
    - Implementar alertas cuando el conteo difiera del esperado
    """)

    return {
        'activos_excel': len(activos_excel),
        'activos_cosmos': len(activos_cosmos),
        'discrepancia': discrepancia
    }

if __name__ == "__main__":
    resultado = analizar_discrepancia()

    print("\n" + "="*80)
    print("📊 RESUMEN")
    print("="*80)
    print(f"Empleados ACTIVOS en Excel:     {resultado['activos_excel']}")
    print(f"Empleados ACTIVOS en Cosmos:    {resultado['activos_cosmos']}")
    print(f"Discrepancia:                   {resultado['discrepancia']}")
    print("="*80)

ANÁLISIS DE DISCREPANCIA - EMPLEADOS ACTIVOS

PASO 1: Analizando Excel del cliente...
Total de empleados en Excel: 300

Valores en campo ESTADO:
ESTADO
ACTIVO      289
INACTIVO     11
Name: count, dtype: int64

Empleados ACTIVOS: 289
Empleados INACTIVOS: 11

PASO 2: Analizando JSON generado (plataforma Narah)...
Empleados ACTIVOS en plataforma: 289
Empleados INACTIVOS en plataforma: 11

PASO 3: Identificando discrepancia...
No hay discrepancia. Conteo coincide.

PASO 4: Buscando causa raíz...

Verificando variaciones en el texto 'ACTIVO':
      'ACTIVO' (normalizado: 'ACTIVO') -> 289 empleados

💡 PASO 6: Propuesta de corrección:

    CAUSA PROBABLE:
    - El pipeline normalizó correctamente el campo ESTADO a mayúsculas
    - Si el Excel tenía variaciones como "Activo ", " ACTIVO", etc., 
      el pipeline las corrigió a "ACTIVO"
    - La discrepancia reportada por el cliente puede deberse a que ellos 
      cuentan manualmente sin normalizar
    
    CORRECCIÓN:
    1. Validar con el c

# **RETO 3**

In [9]:
"""
RETO 3: Correccion de Datos Criticos de Salud

Problemas a resolver:
1. Andres Garcia (10234567): Incapacidad con -45 dias (fechas invertidas)
2. Carlos Niesta (30456789): Codigo CIE10 incorrecto (caso teorico)
3. Crear funcion de validacion reutilizable
"""

import pandas as pd
import os

if not os.path.exists('outputs'):
    os.makedirs('outputs')

# =============================================================================
# FUNCIONES DE VALIDACION
# =============================================================================

def validar_incapacidad(row):
    """
    Valida una incapacidad y retorna lista de errores.

    Detecta:
    - Dias negativos o cero
    - Fecha fin anterior a fecha inicio
    - Codigos CIE10 con formato invalido
    """
    errores = []

    # Regla 1: Dias coherentes
    try:
        dias = int(row['DIAS'])
        if dias <= 0:
            errores.append(f"Dias invalidos: {dias}")
    except:
        errores.append("Dias no es numerico")

    # Regla 2: Fechas logicas
    try:
        fecha_inicio = pd.to_datetime(row['FECHA INICIO INCAPACIDAD'])
        fecha_fin = pd.to_datetime(row['FECHA FIN INCAPACIDAD'])

        if fecha_fin < fecha_inicio:
            errores.append("Fecha fin anterior a fecha inicio")
    except:
        errores.append("Error al procesar fechas")

    # Regla 3: Formato CIE10
    cie10 = str(row['DIAGNOSTICO CIE10']).strip()
    if len(cie10) > 0 and not cie10[0].isalpha():
        errores.append(f"CIE10 '{cie10}' no inicia con letra")

    return errores

# =============================================================================
# CORRECCIONES ESPECIFICAS
# =============================================================================

def corregir_datos_criticos(archivo_excel):
    """
    Aplica correcciones especificas a los casos reportados.

    Returns:
        DataFrame con incapacidades corregidas
    """
    print("="*80)
    print("RETO 3: CORRECCION DE DATOS CRITICOS DE SALUD")
    print("="*80)

    # Cargar datos
    df_incap = pd.read_excel(archivo_excel, sheet_name='Incapacidades')

    # Convertir fechas
    df_incap['FECHA INICIO INCAPACIDAD'] = pd.to_datetime(
        df_incap['FECHA INICIO INCAPACIDAD'], errors='coerce'
    )
    df_incap['FECHA FIN INCAPACIDAD'] = pd.to_datetime(
        df_incap['FECHA FIN INCAPACIDAD'], errors='coerce'
    )

    # --- CORRECCION 1: Caso Andres Garcia - Dias Negativos ---
    print("\nPASO 1: Corrigiendo fechas invertidas")
    print("-"*80)

    # a) Identificar causa raiz
    print("a) Identificando causa raiz...")
    andres = df_incap[df_incap['CEDULA EMPLEADO'] == 10234567]
    if len(andres) > 0:
        print(f"   Empleado: Andres Garcia (10234567)")
        problema = andres[andres['DIAS'] < 0]
        if len(problema) > 0:
            print(f"   Causa raiz: Fechas invertidas")
            print(f"   Fecha inicio: {problema.iloc[0]['FECHA INICIO INCAPACIDAD']}")
            print(f"   Fecha fin:    {problema.iloc[0]['FECHA FIN INCAPACIDAD']}")
            print(f"   Dias:         {problema.iloc[0]['DIAS']}")

    # b) Detectar TODAS las incapacidades con fechas invertidas
    print("\nb) Detectando TODAS las incapacidades con fechas invertidas...")
    mask_fechas_invertidas = df_incap['FECHA FIN INCAPACIDAD'] < df_incap['FECHA INICIO INCAPACIDAD']
    total_invertidas = mask_fechas_invertidas.sum()
    print(f"   Incapacidades con fechas invertidas: {total_invertidas}")

    # c) Ejecutar correccion
    print("\nc) Ejecutando correccion...")
    for idx in df_incap[mask_fechas_invertidas].index:
        inicio_original = df_incap.at[idx, 'FECHA INICIO INCAPACIDAD']
        fin_original = df_incap.at[idx, 'FECHA FIN INCAPACIDAD']

        # Intercambiar fechas
        df_incap.at[idx, 'FECHA INICIO INCAPACIDAD'] = fin_original
        df_incap.at[idx, 'FECHA FIN INCAPACIDAD'] = inicio_original

        # Recalcular dias
        delta = (inicio_original - fin_original).days + 1
        df_incap.at[idx, 'DIAS'] = delta

        print(f"   [CORREGIDO] Fila {idx}: Dias {df_incap.at[idx, 'DIAS']}")

    # --- CORRECCION 2: Caso Carlos Niesta - Codigo CIE10 Erroneo ---
    print("\nPASO 2: Verificando inconsistencias de codigo CIE10")
    print("-"*80)

    # a) Identificar inconsistencia
    print("a) Buscando caso: Carlos Niesta (30456789) con Z000 + FRACTURA...")

    cie10_serie = df_incap['DIAGNOSTICO CIE10'].astype(str).str.strip().str.upper()
    descripcion_serie = df_incap['DESCRIPCION DIAGNOSTICO'].astype(str).str.strip().str.upper()

    # Buscar Z000 con descripcion de fractura
    mask_cie10_error = (
        (cie10_serie.isin(['Z000', 'Z00.0'])) &
        (descripcion_serie.str.contains('FRACTURA', na=False))
    )

    encontrados = df_incap[mask_cie10_error]

    if not encontrados.empty:
        print(f"   [ENCONTRADO] {len(encontrados)} registros con Z000 + FRACTURA")

        # b) Determinar campo correcto
        print("\nb) Determinando campo correcto...")
        print("   Codigo Z000 = Examen medico general (NO es diagnostico de trauma)")
        print("   Descripcion FRACTURA = Trauma fisico")
        print("   CONCLUSION: La descripcion es correcta, el codigo Z000 esta mal")

        # c) Proponer y ejecutar correccion
        print("\nc) Ejecutando correccion...")
        print("   Cambiando Z000 a S729 (Fractura no especificada)")
        df_incap.loc[mask_cie10_error, 'DIAGNOSTICO CIE10'] = 'S729'

        for idx, row in encontrados.iterrows():
            print(f"   [CORREGIDO] Fila {idx}: {row['NOMBRE COMPLETO']}")
    else:
        print("   [INFO] Caso no encontrado en los datos actuales")
        print("   NOTA: El PDF menciona este caso como ejemplo teorico")
        print("         La logica de deteccion y correccion esta implementada")

    return df_incap

# =============================================================================
# VALIDACION COMPLETA
# =============================================================================

def ejecutar_validacion_completa(df):
    """
    Ejecuta validacion sobre todas las incapacidades.
    """
    print("\nPASO 3: Validacion completa de todas las incapacidades")
    print("-"*80)

    errores_encontrados = []

    for idx, row in df.iterrows():
        errores = validar_incapacidad(row)
        if errores:
            errores_encontrados.append({
                'fila': idx,
                'cedula': row['CEDULA EMPLEADO'],
                'errores': ' | '.join(errores)
            })

    print(f"Incapacidades validadas: {len(df)}")
    print(f"Registros con errores: {len(errores_encontrados)}")

    if errores_encontrados:
        print("\nPrimeros 10 errores:")
        for err in errores_encontrados[:10]:
            print(f"  Fila {err['fila']} (Cedula {err['cedula']}): {err['errores']}")

    return errores_encontrados

# =============================================================================
# PROGRAMA PRINCIPAL
# =============================================================================

if __name__ == "__main__":
    archivo ='/content/datos_cliente_bpo_soluciones_1.xlsx'

    try:
        # Ejecutar correcciones
        df_corregido = corregir_datos_criticos(archivo)

        # Validar resultados
        errores = ejecutar_validacion_completa(df_corregido)

        # Guardar archivo corregido
        output_file = 'outputs/incapacidades_corregidas.xlsx'
        df_corregido.to_excel(output_file, index=False)

        print("\n" + "="*80)
        print("RESUMEN")
        print("="*80)
        print(f"Incapacidades procesadas: {len(df_corregido)}")
        print(f"Errores detectados: {len(errores)}")
        print(f"Archivo guardado: {output_file}")
        print("="*80)

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

RETO 3: CORRECCION DE DATOS CRITICOS DE SALUD

PASO 1: Corrigiendo fechas invertidas
--------------------------------------------------------------------------------
a) Identificando causa raiz...
   Empleado: Andres Garcia (10234567)
   Causa raiz: Fechas invertidas
   Fecha inicio: 2024-10-01 00:00:00
   Fecha fin:    2024-09-17 00:00:00
   Dias:         -45

b) Detectando TODAS las incapacidades con fechas invertidas...
   Incapacidades con fechas invertidas: 1

c) Ejecutando correccion...
   [CORREGIDO] Fila 80: Dias 15

PASO 2: Verificando inconsistencias de codigo CIE10
--------------------------------------------------------------------------------
a) Buscando caso: Carlos Niesta (30456789) con Z000 + FRACTURA...
   [INFO] Caso no encontrado en los datos actuales
   NOTA: El PDF menciona este caso como ejemplo teorico
         La logica de deteccion y correccion esta implementada

PASO 3: Validacion completa de todas las incapacidades
--------------------------------------------